In [ ]:
suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(patchwork)
  library(ComplexHeatmap)
  library(circlize)
  library(ggalluvial)
  library(ggrepel)
  library(ggpubr)
  library(grid)
})
root <- getwd()
ctc <- readRDS(file.path(root, "Affirmseq_CTC.rds"))
fd <- ctc@misc$figure_data
tc <- fd$colors$ctc
sc <- fd$colors$sample
theme_set(theme_classic(base_size = 14))


In [ ]:
p_pie <- ggplot(fd$pie, aes("", freq, fill = EMT_by_image)) + geom_col(width = 1, color = "white") + coord_polar(theta = "y") + facet_wrap(~sample_label, ncol = 1) + scale_fill_manual(values = tc, name = "CTC type") + theme_void(base_size = 14) + theme(strip.text = element_text(size = 14), legend.position = "right")
u <- fd$umap
p_sample <- ggplot(u, aes(UMAP_1, UMAP_2, color = sample_label)) + geom_point(size = 2) + scale_color_manual(values = sc, name = NULL) + coord_fixed() + labs(title = "Affirm-seq", x = "UMAP 1", y = "UMAP 2") + theme(plot.title = element_text(hjust = .5))
ggsave(file.path(root, "Affirmseq_CTC_pie_umap.pdf"), p_pie | p_sample, width = 8.2, height = 4.5, device = cairo_pdf)


In [ ]:
m <- fd$classification
cairo_pdf(file.path(root, "Affirmseq_CTC_classification_heatmap.pdf"), width = 5.2, height = 4.2)
draw(Heatmap(m, name = "Proportion", column_title = "Transcriptomic classification", row_title = "Immunofluorescence classification", cluster_rows = FALSE, cluster_columns = FALSE, col = colorRamp2(c(0, .5, 1), c("white", "#fee0d2", "#de2d26")), cell_fun = function(j, i, x, y, w, h, fill) grid.text(sprintf("%.1f%%", m[i, j] * 100), x, y, gp = gpar(fontsize = 12)), border = TRUE))
dev.off()
p_sankey <- ggplot(fd$sankey, aes(axis1 = EMT_by_image, axis2 = EMT_by_gene, y = n)) + geom_alluvium(aes(fill = EMT_by_image), width = 1/12, alpha = .85) + geom_stratum(width = 1/6, fill = "grey90", color = "grey30") + geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3) + scale_x_discrete(limits = c("Immunofluorescence", "Transcriptome"), expand = c(.05, .05)) + scale_fill_manual(values = tc) + labs(x = NULL, y = "Number of cells") + theme_minimal(base_size = 12) + theme(panel.grid = element_blank(), axis.text.y = element_blank(), axis.ticks = element_blank(), legend.position = "none")
u$EMT_plot <- ifelse(is.na(u$EMT_by_image), "Unknown", as.character(u$EMT_by_image))
p_type <- ggplot(u, aes(UMAP_1, UMAP_2)) + geom_point(data = subset(u, EMT_plot == "Unknown"), color = "grey85", size = .8) + geom_point(data = subset(u, EMT_plot != "Unknown"), aes(color = EMT_plot), size = 1.8) + scale_color_manual(values = tc, name = "CTC type") + coord_fixed() + labs(x = "UMAP 1", y = "UMAP 2")
p_scatter <- ggplot(fd$chol_data, aes(EMT_score, Chol_AUC, color = sample_type2)) + geom_point(size = 2.2, alpha = .85) + geom_smooth(method = "lm", color = "black", fill = "grey85") + facet_wrap(~sample_type2, labeller = as_labeller(c(BCR = "BCR", Met_1 = "Met 1", Met_2 = "Met 2"))) + scale_color_manual(values = fd$colors$sample2) + labs(x = "EMT score (Transcriptomic)", y = "Cholesterol AUC score") + guides(color = "none") + theme_bw(base_size = 12)
ggsave(file.path(root, "Affirmseq_CTC_classification_sankey_umap_scatter.pdf"), p_sankey | p_type | p_scatter, width = 12, height = 4.2, device = cairo_pdf)


In [ ]:
d <- fd$deg_rna
p_vol <- ggplot(d, aes(avg_log2FC, -log10(p_val_adj), color = change)) + geom_point(size = 2.2, alpha = .9) + geom_vline(xintercept = c(-.25, .25), linetype = 4, color = "grey40") + geom_hline(yintercept = -log10(.05), linetype = 4, color = "grey40") + geom_text_repel(data = subset(d, label != ""), aes(label = label), size = 3, color = "black", max.overlaps = 50) + scale_color_manual(values = c("Up-regulated (mCTCs)" = "#E74C3C", "Down-regulated (mCTCs)" = "#377EB8", Stable = "lightgrey"), name = NULL) + labs(x = expression(log[2]~"(Fold change)"), y = expression(-log[10]~"(Adjusted "*italic(P)*"-value)"))
cd <- fd$chol_data
cd$EMT_by_image <- factor(cd$EMT_by_image, c("Epithelial", "Hybrid", "Mesenchymal"))
p_chol <- ggbarplot(cd, x = "EMT_by_image", y = "Chol_AUC", fill = "EMT_by_image", add = c("mean_se", "jitter"), add.params = list(alpha = .45, size = .9, width = .25, color = "black"), facet.by = "sample_type2", palette = tc, legend = "none", ylab = "AUCell score", xlab = "") + stat_compare_means(comparisons = list(c("Epithelial", "Hybrid"), c("Hybrid", "Mesenchymal"), c("Epithelial", "Mesenchymal")), method = "wilcox.test", label = "p.format", size = 3) + stat_compare_means(method = "kruskal.test", size = 3) + theme(axis.text.x = element_text(angle = 45, hjust = 1))
ggsave(file.path(root, "Affirmseq_CTC_volcano_cholesterol.pdf"), p_vol | p_chol, width = 11, height = 4.5, device = cairo_pdf)


In [ ]:
cairo_pdf(file.path(root, "Affirmseq_CTC_pathway_heatmap.pdf"), width = 4.8, height = 4)
draw(Heatmap(fd$target_heatmap, name = "Z-score", col = colorRamp2(c(-2, 0, 2), c("#5DADE2", "white", "#E74C3C")), cluster_columns = FALSE, cluster_rows = TRUE, rect_gp = gpar(col = "white", lwd = 1), column_names_gp = gpar(fontsize = 11), row_names_gp = gpar(fontsize = 8)))
dev.off()
ma <- fd$metabolism_anno
left <- rowAnnotation(Sample = ma$Sample, `CTC type` = ma$CTC_type, col = list(Sample = c(BCR = "#EF787E", Met_1 = "#85C1E9", Met_2 = "#2E86C1"), `CTC type` = tc))
cairo_pdf(file.path(root, "Affirmseq_CTC_metabolism_heatmap.pdf"), width = 6.2, height = 4.5)
draw(Heatmap(fd$metabolism_heatmap, name = "Z-score", left_annotation = left, col = colorRamp2(c(-2, 0, 2), c("#5DADE2", "white", "#E74C3C")), row_split = ma$Sample, cluster_rows = FALSE, cluster_columns = TRUE, rect_gp = gpar(col = "white", lwd = 1), border = TRUE, column_names_rot = 45, row_names_gp = gpar(fontsize = 8), column_names_gp = gpar(fontsize = 8)))
dev.off()
ca <- fd$cell_pathway_anno
ra <- rowAnnotation(`CTC type` = ca$EMT_by_image, `Sample type` = ca$sample_type, Sample = ca$sample_label, col = list(`CTC type` = tc, `Sample type` = c(BCR = "#984EA3", Met = "#FF7F00"), Sample = sc))
cairo_pdf(file.path(root, "Affirmseq_CTC_cell_pathway_heatmap.pdf"), width = 9, height = 5.8)
draw(Heatmap(fd$cell_pathway_heatmap, name = "Z-score", left_annotation = ra, col = colorRamp2(c(-2, 0, 2), c("#5DADE2", "white", "#E74C3C")), row_split = ca$sample_type, cluster_rows = TRUE, cluster_columns = TRUE, show_row_names = FALSE, row_names_gp = gpar(fontsize = 6), column_names_gp = gpar(fontsize = 6), column_names_rot = 90, use_raster = TRUE))
dev.off()
